In [ ]:

import os
import time
import pandas as pd
import sys
import copy
from torch.utils.data import DataLoader
from datetime import datetime
import pytz

from data_loader import allocate_malicious_nodes
from data_loader import generate_topology
from main_cnn_GPU import run_simulation_CNN_GPU
from data_loader import set_seed
from data_loader import distribute_data
from data_loader import get_data
from theoretical_intensity import calculate_theoretical_intensity




original_stdout = sys.stdout
original_stderr = sys.stderr
class DualLogger(object):
    def __init__(self, file_path):
        self.terminal = sys.stdout
        self.log = open(file_path, "a", encoding='utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()

    def close(self):
        self.log.close()

import sys
# ==========================================
# 1.Experiment settup
# ==========================================



NUM_CLIENTS = 20
BOOST_FACTORS = 2
MALICIOUS_RATIO = 0.3
defense_budget  = int(NUM_CLIENTS * 0.2)
GLOBAL_ROUNDS = 15
EPOCHS = 5
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0

SCALE_FACTOR = 0.8
train_ds, test_ds = get_data()
test_loader = DataLoader(test_ds, batch_size=256)


SEEDS = [1,2,3]
MECHANISM_LIST = ['MAB',  'FLAME','CosL2', 'FedAvg','TrimmedMean', 'Krum']
MAL_RATIOS = [ 0.3, 0.1, 0.2]
TOPO_TYPES = [ 'scale_free']
DEFENSE_RATIOS = [ 0.2]

NUM_CLIENTS = 20
GLOBAL_ROUNDS = 15
NORM_FACTOR = 1
SCALE_FACTOR = 0.8
ATK_TYPE = 'neurotoxin'
INTENSITY = 4.0
# ==========================================
# 2
# ==========================================

SAVE_PATH = ''

for topo_type in TOPO_TYPES:
    for mal_ratio in MAL_RATIOS:
        for def_ratio in DEFENSE_RATIOS:


            num_mal = int(NUM_CLIENTS * mal_ratio)

            current_def_budget = int(NUM_CLIENTS * def_ratio)
            for seed_val in SEEDS:


                print(f"\n{'='*60}")
                print(f"⏰ : {time.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"📡 : Topo={topo_type}, Mal={mal_ratio}, Def={def_ratio}")
                print(f"{'='*60}")
                print(f"\n{'#'*60}")
                print(f"📡 Topo: {topo_type} | Mal: {mal_ratio} | Def: {def_ratio} | Seed: {seed_val}")
                print(f"{'#'*60}")

                set_seed(seed_val)
                client_datasets = distribute_data(train_ds, NUM_CLIENTS)
                G = generate_topology(NUM_CLIENTS, topo_type)
                neighbors = {node: list(G.neighbors(node)) for node in G.nodes()}


                malicious_clients, defense_nodes = allocate_malicious_nodes(
                    G, num_mal, current_def_budget, topology_type=topo_type,placement='Topology-Aware',# placement='Topology-Aware'
                )

                theo_intensities = calculate_theoretical_intensity(
                    neighbors, malicious_clients, NUM_CLIENTS, BOOST_FACTORS, lambda_benign=0.3
                )

                for mech in MECHANISM_LIST:
                    all_results = []
                    csv_filename = f"Final_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.csv"
                    full_save_path = os.path.join(SAVE_PATH, csv_filename)
                    log_filename = f"Log_CNN_{topo_type}_MR{mal_ratio}_DR{def_ratio}_{mech}_seed{seed_val}.txt"
                    log_full_path = os.path.join(SAVE_PATH, log_filename)

                    if os.path.exists(full_save_path):
                        print(f"⏩Eisting file: {csv_filename}")
                        continue
                    logger = DualLogger(log_full_path)
                    sys.stdout = logger
                    sys.stderr = logger
                    shanghai_tz = pytz.timezone('Asia/Shanghai')
                    print(f"🚀 Running: {mech}")
                    start_wall_time = datetime.now(shanghai_tz).strftime("%Y-%m-%d %H:%M:%S")
                    start_tick = time.time()

                    _, _, accs, asrs = run_simulation_CNN_GPU(
                        seed_val, NUM_CLIENTS, defense_nodes, malicious_clients,
                        G, neighbors, client_datasets, test_loader,
                        atk_type=ATK_TYPE,
                        mechanism=mech,
                        intensity=INTENSITY,
                        norm_factor=NORM_FACTOR,
                        debug_mode=False,
                        GLOBAL_ROUNDS=GLOBAL_ROUNDS,
                        epochs=5, debug=False
                    )
                    end_tick = time.time()
                    duration_sec = round(end_tick - start_tick, 2)

                    result_entry_base = {
                        'seed': seed_val,
                        'mechanism': mech,
                        'malicious_ratio': mal_ratio,
                        'defense_ratio': def_ratio,
                        'topology': topo_type,
                        'norm_factor': NORM_FACTOR,
                        'scale_factor': SCALE_FACTOR,
                        'global_rounds': GLOBAL_ROUNDS,
                        'start_time': start_wall_time,
                        'duration_sec': duration_sec
                    }

                    for i in range(NUM_CLIENTS):
                        client_row = copy.deepcopy(result_entry_base)
                        client_row.update({
                            'client_id': i,
                            'final_acc': accs[i],
                            'final_asr': asrs[i],
                            'theo_intensity': theo_intensities[i] if i < len(theo_intensities) else 0.0,
                            'node_type': 'MAL' if i in malicious_clients else ('DEF' if i in defense_nodes else 'BEN')
                        })
                        all_results.append(client_row)

                    pd.DataFrame(all_results).to_csv(full_save_path, index=False)


                    sys.stdout = original_stdout

                    sys.stdout = logger

print(f"\n🎉 Experiments compeleted！")

In [ ]:
#Table 2
import os
import glob
import pandas as pd

# ==========================================
#
# ==========================================
SAVE_PATH = ''
search_pattern = os.path.join(SAVE_PATH, "Final_CNN*.csv")
all_files = glob.glob(search_pattern)

list_df = []

for f in all_files:
    temp_df = pd.read_csv(f)

  
    if "placement_random" in os.path.basename(f):
        temp_df['mechanism'] = 'MAB(RANDOM TOPY)'
        list_df.append(temp_df)

   
    else:
        temp_df['mechanism'] = temp_df['mechanism'].replace({'MAB': 'MAB(TOPY AWARE)'})
        list_df.append(temp_df)


df_results = pd.concat(list_df, ignore_index=True)


df_results['Role'] = df_results['node_type'].apply(lambda x: 'Malicious' if x == 'MAL' else 'Benign')
df_benign = df_results[df_results['Role'] == 'Benign']


benign_comparison = df_benign.pivot_table(
    index='mechanism',
    columns='malicious_ratio',
    values=['final_acc', 'final_asr'],
    aggfunc='mean'
)

print(benign_comparison)


# ==========================================
#  
# ==========================================

# 1.
df_results2 = df_results[df_results['topology'] == 'scale_free'].copy()#'scale_free' 'random_regular'
df_results2 = df_results2[df_results2['malicious_ratio'] != 0.4].copy()
df_benign = df_results2[df_results2['Role'] == 'Benign'].copy()

# 2. 
trial_means = df_benign.groupby(
    ['mechanism', 'malicious_ratio', 'seed']
)[['final_acc', 'final_asr']].mean().reset_index()

# 3. 
stats = trial_means.groupby(
    ['mechanism', 'malicious_ratio']
)[['final_acc', 'final_asr']].agg(['mean', 'std']).reset_index()

# 4. 
def format_mean_std(row, metric):
    m = row[(metric, 'mean')]
    s = row[(metric, 'std')]
    return f"{m:.2f} ({s:.2f})"

stats['ACC'] = stats.apply(lambda r: format_mean_std(r, 'final_acc'), axis=1)
stats['ASR'] = stats.apply(lambda r: format_mean_std(r, 'final_asr'), axis=1)

# 5. 
clean_stats = stats[['mechanism', 'malicious_ratio', 'ACC', 'ASR']].copy()
latex_table = clean_stats.pivot(index='mechanism', columns='malicious_ratio', values=['ACC', 'ASR'])

# 6. 
latex_table = latex_table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)


def format_mean_std_percent(row, metric):
    m = row[(metric, 'mean')]
    s = row[(metric, 'std')]

    return f"{m:05.2f} ({s:05.2f})"

# 
stats['ACC (%)'] = stats.apply(lambda r: format_mean_std_percent(r, 'final_acc'), axis=1)
stats['ASR (%)'] = stats.apply(lambda r: format_mean_std_percent(r, 'final_asr'), axis=1)

# 
clean_stats = stats[['mechanism', 'malicious_ratio', 'ACC (%)', 'ASR (%)']].copy()
latex_table = clean_stats.pivot(index='mechanism', columns='malicious_ratio', values=['ACC (%)', 'ASR (%)'])

# 
latex_table = latex_table.swaplevel(0, 1, axis=1).sort_index(axis=1, level=0)

# 
custom_order = [
    'FedAvg',
    'Krum',
    'TrimmedMean',
    'CosL2',
    'FLAME',
    'MAB(RANDOM TOPY)', 
    'MAB(TOPY AWARE)',
]
# 
existing_order = [m for m in custom_order if m in latex_table.index]
other_mechs = [m for m in latex_table.index if m not in custom_order]
final_order = existing_order + other_mechs

latex_table = latex_table.reindex(final_order)

# 
try:
    # 
    latex_output = latex_table.style.format(escape="latex").to_latex(
        hrules=True,
        multicol_align="c",
        caption="Macro-Averaged Comparative Performance Summary (Random Regular)",
        label="tab:macro_results_random_regular"
    )
except AttributeError:
    # 
    latex_output = latex_table.to_latex(
        multicolumn=True,
        multicolumn_format='c',
        booktabs=True,
        escape=False,
        caption="Macro-Averaged Comparative Performance Summary (Random Regular)",
        label="tab:macro_results_random_regular"
    )

print(latex_output)


In [ ]:
#Figure 2b
import matplotlib.pyplot as plt
import seaborn as sns
# ==========================================
# 
# ==========================================

if not df_results.empty and 'duration_sec' in df_results.columns:
   

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 6))

    all_mechanisms = df_results['mechanism'].unique().tolist()

    target_first = 'MAB(TOPY AWARE)'

    if target_first in all_mechanisms:
        custom_order = [target_first] + sorted([m for m in all_mechanisms if m != target_first])
    else:
       
        custom_order = sorted(all_mechanisms)

 
    sns.barplot(
        data=df_results,
        x='mechanism',
        y='duration_sec',
        order=custom_order,
        palette='viridis',
        capsize=.1,
        errorbar='sd'
    )

    plt.title('Average Duration by Mechanism (Transformer)', fontsize=14)
    plt.xlabel('Mechanism', fontsize=12)
    plt.ylabel('Duration (seconds)', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show() 